# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
# Imports
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'
OLLAMA_BASE_URL = 'http://localhost:11434/v1'

In [3]:
# set up environment

load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

# OpenAI client
openai_client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

# Model choices: gpt-4o-mini and Llama only (OpenRouter model IDs)
MODELS = {
    "gpt-4o-mini": "openai/gpt-4o-mini",
    "Llama": "ollama/llama3.2",
}

DEFAULT_MODEL = "gpt-4o-mini"

In [4]:
SYSTEM_PROMPT = """
You are an expert indie game designer and technical game architect.

Your role is to generate practical, implementable game ideas for a solo developer.

Guidelines:
- Focus on core gameplay mechanics and game loops.
- Avoid vague descriptions.
- Keep art requirements minimal unless explicitly requested.
- Suggest systems that are technically feasible for a small team or solo developer.
- When appropriate, describe how the game could be structured in code (systems, state management, logic flow).
- Prioritize replayability and retention.
- Be concise but clear.
- Use structured sections when explaining ideas.

Always structure your answers like this:

1. Game Concept (1–2 sentences)
2. Core Loop
3. Core Mechanics
4. Progression System
5. Technical Implementation Notes
6. Why This Is Good For a Solo Developer
"""

In [5]:
# Design reference: short notes the model can pull when discussing a term
DESIGN_REF = {
    "core loop": "Core loop: the repeatable cycle of actions that defines the game (e.g. explore → fight → loot → upgrade → repeat). Keep it tight for solo dev.",
    "roguelike": "Roguelike: permadeath, procedural levels, often turn-based. Great for replayability and low asset count; focus on systems over content.",
    "metroidvania": "Metroidvania: interconnected world, ability-gated progression, backtracking. Plan your map and lock/key abilities early.",
    "procgen": "Procedural generation: levels or content from algorithms. Reduces art burden; balance with seeds and tuning knobs for feel.",
    "state machine": "State machine: game or entity has discrete states (idle, attack, hurt). Clean for AI and animation; easy to implement and debug.",
    "game jam": "Game jam: short, scoped event (e.g. 48h). Constrains scope and forces a clear core loop—ideal for solo practice.",
    "retention": "Retention: what brings players back (daily goals, meta-progression, unlocks). Even small games benefit from a simple progression hook.",
    "solo developer": "Solo dev: scope small, reuse systems, avoid custom art where possible. Prototype the core loop first; cut everything else.",
}

def get_design_ref(term: str) -> str:
    key = (term or "").lower().strip()
    return DESIGN_REF.get(key, f"No cached note for '{term}'. Rely on your expertise and keep it brief for a solo dev.")

def design_ref_tool():
    """OpenAI-style tool spec for the design reference lookup."""
    return {
        "type": "function",
        "function": {
            "name": "get_design_ref",
            "description": "Fetch a short reference note for a game design term (core loop, roguelike, metroidvania, procgen, state machine, game jam, retention, solo developer). Use to back up your answer when relevant.",
            "parameters": {
                "type": "object",
                "properties": {"term": {"type": "string", "description": "Game design term to look up"}},
                "required": ["term"],
                "additionalProperties": False,
            },
        },
    }

TOOLS = [design_ref_tool()]

In [6]:
def assistant_as_api_message(msg):
    """Turn the assistant message (with optional tool_calls) into the dict format the API expects."""
    return {
        "role": "assistant",
        "content": msg.content or "",
        "tool_calls": [
            {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in msg.tool_calls
        ],
    }

def run_tools_and_collect(assistant_msg):
    """Run any tool calls in the assistant message and return the list of tool response dicts."""
    out = []
    for tc in assistant_msg.tool_calls:
        name = tc.function.name
        try:
            args = json.loads(tc.function.arguments or "{}")
        except json.JSONDecodeError:
            args = {}
        if name == "get_design_ref":
            text = get_design_ref(args.get("term", ""))
        else:
            text = f"Unknown tool: {name}"
        out.append({"role": "tool", "content": text, "tool_call_id": tc.id})
    return out

In [7]:
def build_conversation(history, latest_user_text):
    """Conversation as API messages: system + history + latest user."""
    flat = [{"role": m.get("role", "user"), "content": m.get("content", "") or ""} for m in (history or [])]
    return [{"role": "system", "content": SYSTEM_PROMPT}] + flat + [{"role": "user", "content": latest_user_text}]

def _yield_in_chunks(full_text, chunk_size=8):
    """Yield full_text in small chunks so the UI can show progressive updates."""
    acc = ""
    for i in range(0, len(full_text), chunk_size):
        acc = full_text[: i + chunk_size]
        yield acc
    if len(acc) < len(full_text):
        yield full_text

def reply_with_tools(model_id, messages):
    """Call model with tools; run tool calls and re-call until we get a text reply. Yields in chunks so UI streams."""
    resp = openai_client.chat.completions.create(model=model_id, messages=messages, tools=TOOLS)
    while getattr(resp.choices[0], "finish_reason", None) == "tool_calls":
        msg = resp.choices[0].message
        messages.append(assistant_as_api_message(msg))
        messages.extend(run_tools_and_collect(msg))
        resp = openai_client.chat.completions.create(model=model_id, messages=messages, tools=TOOLS)
    content = resp.choices[0].message.content or ""
    for chunk in _yield_in_chunks(content):
        yield chunk

def reply_streaming(model_id, messages):
    """Stream model reply using OpenAI-compatible streaming (OpenRouter)."""
    stream = openai_client.chat.completions.create(
        model=model_id, messages=messages, stream=True
    )
    acc = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        part = chunk.choices[0].delta.content
        if part:
            acc += part
            yield acc

def generate_reply(user_text, history, model_name):
    """Single entry point: build messages, pick model, then either run tools or stream."""
    messages = build_conversation(history, user_text)
    model_id = MODELS.get(model_name, MODELS[DEFAULT_MODEL])
    use_tools = not model_id.startswith("ollama/")

    if use_tools:
        for content in reply_with_tools(model_id, messages):
            yield content
    else:
        for content in reply_streaming(model_id, messages):
            yield content

In [8]:
# UI: helpers and layout
def as_display_msg(item):
    """Normalize a history item to {role, content} for the chatbot."""
    if isinstance(item, dict):
        return {"role": item.get("role", "user"), "content": item.get("content", "") or ""}
    return {"role": "user", "content": str(item)}

def append_question_and_clear(question, chat_history):
    """Add the user question to chat history and clear the input. Returns (empty input, updated history)."""
    chat_history = list(chat_history or [])
    if question:
        chat_history.append({"role": "user", "content": question})
    return "", chat_history

def stream_reply_into_history(chat_history, model_name):
    """Take the last user message from history, get model reply, yield full history + growing assistant text."""
    chat_history = list(chat_history or [])
    if not chat_history:
        return
    last = as_display_msg(chat_history[-1])
    if last["role"] != "user":
        return
    user_text = last["content"]
    prev_turns = [as_display_msg(h) for h in chat_history[:-1]]
    displayed = [as_display_msg(h) for h in chat_history]
    for content in generate_reply(user_text, prev_turns, model_name):
        yield displayed + [{"role": "assistant", "content": content}]

with gr.Blocks(title="Indie Game Design Q&A") as app:
    gr.Markdown("Ask about game design, mechanics, or solo dev. Try: *Design a roguelike* or *Core loop for a puzzle game*.")
    chat_log = gr.Chatbot(height=500, type="messages")
    model_selector = gr.Dropdown(choices=list(MODELS.keys()), value=DEFAULT_MODEL, label="Model")
    with gr.Row():
        question_input = gr.Textbox(label="Your question", placeholder="Ask a game design question...", scale=4)
        reset_btn = gr.Button("Clear", scale=1)
    send_btn = gr.Button("Submit", variant="primary")

    def reset_chat():
        return [], ""

    question_input.submit(append_question_and_clear, [question_input, chat_log], [question_input, chat_log]).then(
        stream_reply_into_history, [chat_log, model_selector], chat_log
    )
    send_btn.click(append_question_and_clear, [question_input, chat_log], [question_input, chat_log]).then(
        stream_reply_into_history, [chat_log, model_selector], chat_log
    )
    reset_btn.click(reset_chat, None, [chat_log, question_input])

In [ ]:
app.launch()